# Manuscript trajectory figures and SI analyses
This output-free notebook exposes the supported numerical and plotting inputs for the manuscript trajectory panels using local-coordinate reconstructed paths. It does not write figures or outputs.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from trajectory.manuscript import manuscript_summary, phase_summary, representative_path, surface_threshold_sensitivity, pooled_steps, buzz_centred_results, plot_buzz_centred_displacement, plot_manuscript_panels
from trajectory.analysis import fit_powerlaw
INDEX = ROOT / 'data/dswp/trajectory_index.csv'
EVENTS = pd.read_csv(ROOT / 'data/dswp/buzz_events.csv')
summary = manuscript_summary(INDEX)
summary['index_csv'] = INDEX
summary['population']

The pooled step-length tail and per-dive/per-individual summaries below are the release's descriptive trajectory results. The fitted exponent is called `mu` in the manuscript code and should be described as Lévy-like rather than as proof of a Lévy search process.

In [ ]:
steps = pooled_steps(INDEX)
pooled_fit = fit_powerlaw(steps)
per_dive = summary['per_dive']
per_individual = summary['per_individual']
pooled_fit, per_dive[['trajectory_id','mu','alpha_msd']].head(), per_individual.head()

In [ ]:
meta, xyz = representative_path(INDEX)
phase = phase_summary(INDEX)
meta[['trajectory_id','individual_id']], phase.head(), xyz[:3]

The phase table uses the reviewed foraging bounds already present in `trajectory_index.csv`. Temporal-scale MSD is returned by the production analysis; the surface-threshold sweep tests the manuscript's 5, 10, 20, 30, 50, and 100 m boundaries without changing the released paths.

In [ ]:
timescale = summary['timescale']
surface = surface_threshold_sensitivity(INDEX)
timescale.filter(regex='alpha_').median(), surface.groupby('threshold_m').alpha_msd.median()

## Public-data boundaries and buzz-centred audit

Known issue touchpoint: location-bearing map limitations. The release contains translated local coordinates only, so regional maps, bathymetry, vessel tracks, and identified-trajectory maps are intentionally blocked here. No location data are invented.

Known issue touchpoint resolved: buzz-centred time origin and labels. The manuscript-producing notebook subtracted `dive_start` from buzz times that were already dive-local. The manuscript-facing result now uses dive-local times and the registered GMM, retains 20 whales, and applies Bonferroni correction across eight positive lags. The archived-label and wrong-time paths remain below only to document why the former eight-whale null result occurred.

In [ ]:
legacy_labels = pd.read_csv(ROOT / 'data/dswp/comeback_legacy_labels.csv')
buzz_msd_corrected = buzz_centred_results(EVENTS, INDEX, time_origin='dive_local', window_s=30, minimum_per_type=5)
buzz_msd_legacy_labels_corrected_time = buzz_centred_results(EVENTS, INDEX, labels=legacy_labels, label_column='legacy_buzz_type', time_origin='dive_local', window_s=30, minimum_per_type=5)
buzz_msd_forensic_legacy = buzz_centred_results(EVENTS, INDEX, labels=legacy_labels, label_column='legacy_buzz_type', time_origin='legacy_subtract_dive_start', window_s=30, minimum_per_type=5)
{'registered_corrected_n_individuals': buzz_msd_corrected['n_individuals'], 'registered_corrected_buzz_counts': buzz_msd_corrected['retained_buzz_counts'], 'legacy_labels_corrected_time_n_individuals': buzz_msd_legacy_labels_corrected_time['n_individuals'], 'forensic_legacy_n_individuals': buzz_msd_forensic_legacy['n_individuals']}, buzz_msd_corrected['paired_tests'], buzz_msd_legacy_labels_corrected_time['paired_tests'], buzz_msd_forensic_legacy['paired_tests']

In [ ]:
plots = plot_manuscript_panels(summary, xyz, surface, buzz_msd_corrected)
buzz_displacement_axes = plot_buzz_centred_displacement(buzz_msd_corrected)
plots['figure'], buzz_displacement_axes[0].figure

Known issue touchpoint: reconstruction wording. These analyses begin from released reconstructed positions, not raw sensor channels.

Known issue touchpoint: Levy fitter naming. The tail fit and MSD are descriptive diagnostics; they do not identify a unique generative model.

No cells write files, save figures, or retain execution counts.